# Descrição geral
Os dados foram sintetisados usando um script simples, aqui o foco é gerar um conjunto mais simples, mais rapido e mais facil de usar no treinamento

In [ ]:
from seeds.seed_perfil_usuario_kmeans import gerar_tabela
df_perfil_usuario = gerar_tabela(numero_linhas_iguais=60000, numero_linhas_abaixo=20000, numero_linhas_acima=20000)

In [ ]:
print('Trechos')
display(df_perfil_usuario.head())

print('Shape')
print(df_perfil_usuario.shape)

print('Potenciais falhas')
print(df_perfil_usuario.isnull().sum())

# Normalizando valores

In [ ]:
from sklearn.preprocessing import StandardScaler
import pandas as pd

scaler = StandardScaler()

# fit + transform: aprende a média/desvio e aplica
X_normalizado = scaler.fit_transform(df_perfil_usuario)

# X_normalizado é um numpy array, média 0 e desvio 1 em cada coluna
print("Média após normalização:", X_normalizado.mean(axis=0).round(2))
print("Desvio após normalização:", X_normalizado.std(axis=0).round(2))

df_normalizado = pd.DataFrame(X_normalizado, columns=df_perfil_usuario.columns.to_list())

display(df_normalizado.head())

# Escolher quantidade de grupos

A quantidade de grupos já foi escolhida no primeiro treino, a celula vai continuar no codigo para mostrar que o valor escolhido não foi aleatorio

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

inercias = []
grupos_teste = range(2, 20)

for k in grupos_teste:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_normalizado)
    inercias.append(kmeans.inertia_)
    print(f"Grupos: {k} | Inércia: {kmeans.inertia_:.2f}")


plt.figure(figsize=(8, 5))
plt.plot(grupos_teste, inercias, marker='o', linestyle='-', color='steelblue')
plt.title('Método do Cotovelo')
plt.xlabel('Número de grupos (k)')
plt.ylabel('Inércia (WCSS)')
plt.xticks(grupos_teste)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Treino do modelo
aqui é onde o filho chora e a mãe não vê

In [ ]:
from sklearn.cluster import KMeans
import pandas as pd


kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
kmeans.fit(X_normalizado)

print(f"Modelo treinado com {kmeans.n_clusters} grupos")
print(f"Inércia final: {kmeans.inertia_:.2f}")

# Desnormalizar os centros para ler em percentuais
centros = pd.DataFrame(
    kmeans.cluster_centers_,
    columns=df_normalizado.columns.to_list()
)

display(centros)


In [ ]:
rotulos = kmeans.labels_
contagem = pd.Series(rotulos).value_counts().sort_index()
print("\n" + "="*70)
print("USUÁRIOS POR PERFIL")
print("="*70)
print(contagem)


In [ ]:
# Mensagens por categoria
ALERTAS = {
    "ALIMENTACAO": "Você gasta muito com alimentação. Tente reduzir refeições fora de casa.",
    "TRANSPORTE": "Gastando muito com transporte. Tente meios alternativos (ônibus, bike, carona).",
    "SAUDE": "Gastos com saúde acima do perfil ideal. Verifique se todos são necessários.",
    "MORADIA": "Moradia está pesando no orçamento. Avalie se compensa a localização/custo.",
    "EDUCACAO": "Investimento em educação alto. Certifique-se de que está gerando retorno.",
    "LAZER": "Lazer acima do ideal do seu perfil. Reduza passeios/eventos por um tempo.",
    "SERVICOS": "Serviços diversos acima do padrão. Revise contratos e taxas.",
    "ASSINATURAS": "Muitas assinaturas. Cancele as que não usa.",
    "DIVIDAS": "Dívidas acima do ideal. Priorize quitar as com juros mais altos.",
    "POUPANCA": "Poupança abaixo do ideal. Tente guardar um pouco mais todo mês."
}

def alertas_usuario(dados_usuario: pd.DataFrame, limiar: float = 15):
    """
    Retorna lista de alertas (multi-label) para o usuário.
    """
    colunas = dados_usuario.columns.tolist()
    
    X_norm = scaler.transform(dados_usuario)
    cluster = kmeans.predict(X_norm)[0]
    
    centro_norm = kmeans.cluster_centers_[cluster]
    
    centro_original = scaler.inverse_transform(centro_norm.reshape(1, -1))[0]
    usuario_original = dados_usuario.values[0]
    
    alertas = []
    for i, col in enumerate(colunas):
        diff_pct = ((usuario_original[i] - centro_original[i]) / (centro_original[i] + 1e-6)) * 100
        
        if diff_pct > limiar and col in ALERTAS:
            alertas.append({
                "categoria": col,
                "diferenca_pct": round(diff_pct, 1),
                "mensagem": ALERTAS[col]
            })
    
    return {
        "cluster": int(cluster),
        "alertas": alertas,
        "total_alertas": len(alertas)
    }


# ========== USO ==========

novo = pd.DataFrame([{
    "ALIMENTACAO": 100.0,
    "TRANSPORTE": 100.0,
    "SAUDE": 100.0,
    "MORADIA": 100.0,
    "EDUCACAO": 100.0,
    "LAZER": 100.0,
    "SERVICOS": 100.0,
    "ASSINATURAS": 100.0,
    "DIVIDAS": 100.0,
    "POUPANCA": 100.0
}])

resultado = alertas_usuario(novo)
print(f"Cluster: {resultado['cluster']} | Alertas: {resultado['total_alertas']}")
for a in resultado['alertas']:
    print(f"  [{a['categoria']}] +{a['diferenca_pct']}% → {a['mensagem']}")

# Exportar modelo


In [ ]:
import joblib

joblib.dump(kmeans, "modelos/kmeans_perfil_sugestao.joblib")
joblib.dump(scaler, "modelos/scaler_perfil_sugestao.joblib")